# Evaluated application release
This notebook reads actual release evidence; it does not train on test rows. The bundled model is scoped to independent synthetic Docker runs. External network validation remains required. Rebuild a release with `tools/collect_sessions.py` and `tools/train_release.py`, using a new release directory and fresh final-test captures.

In [ ]:
import json, sys
from pathlib import Path
working = Path.cwd().resolve()
root = next(candidate for base in (working, *working.parents) for candidate in (base, base / 'network-anomaly-detection') if (candidate / 'src/release.py').is_file())
sys.path.insert(0, str(root))
from src.release import ReleaseModel
from src.streaming import replay_capture
model = ReleaseModel.load_default()
print(model.version, model.manifest['scope'])
print(model.manifest['provenance']['chosen_candidate'])

In [ ]:
import pandas as pd
evaluation = model.manifest['evaluation']
display(pd.DataFrame(evaluation['categories']).T)
print('FPR:', evaluation['false_positive_rate'])
print('95% interval:', evaluation['false_positive_rate_95pct_wilson'])
display(pd.DataFrame(evaluation['sessions']).T[['flows','false_positive_rate','recall']])

## Replay a retained capture
Set `capture` to a PCAP from your own lab. The same bounded flow engine and causal source-start window are used by the live sensor. IPv4 TCP/UDP are supported. The feature version is checked during inference.

```python
capture = root.parent / 'artifacts/sessions/your-run.pcap'
flows = replay_capture(capture, model.flow_config)
verdicts = model.predict(flows)
display(verdicts.head())
```

Capture-wide rate calculations and the old `rename_map` are deliberately absent. A low laboratory false-alarm estimate is not a production guarantee.